|<img src="https://www.udp.cl/cms/wp-content/uploads/2021/06/UDP_LogoRGB_2lineas_Color_SinFondo.png" width="500" height="100">| <p> Ciencias de Datos para la Economía</p>  <p> Ingeniería Comercial </p><p> Profesor: Luis Cuevas Parra </p><p>Unidad I: Entorno, kit de inicio y primer commit</p>|
|--- | :--- |

**Ciencia de Datos para Economia (ICO09425)** · jueves 6 de agosto de 2026

Al terminar este cuaderno debes tener:

1. el entorno de Python verificado, con los paquetes del curso disponibles;
2. el kit de inicio corriendo en **modo demo**, sin necesidad de credenciales;
3. la base `coyuntura.db` creada, con sus tres tablas pobladas;
4. una primera consulta SQL ejecutada sobre esa base;
5. el repositorio del equipo creado y tu primer commit registrado.

> Ejecuta las celdas **en orden**. Si una falla, no sigas: anota el mensaje completo y consulta. Un error temprano ignorado se convierte en tres errores despues.

---
## 1. Verificar el entorno

Antes de instalar nada, veamos con que contamos.

In [1]:
import sys
import platform

print("Python :", sys.version.split()[0])
print("Sistema:", platform.system(), platform.release())

assert sys.version_info >= (3, 9), "Se requiere Python 3.9 o superior"
print("\nVersion de Python correcta.")

Python : 3.13.9
Sistema: Windows 11

Version de Python correcta.


Ahora los paquetes. El nombre con que se importa un paquete no siempre coincide con el nombre con que se instala: `sklearn` se instala como `scikit-learn`, y `dotenv` como `python-dotenv`.

In [2]:
import importlib

# (nombre para importar, nombre para instalar)
PAQUETES = [
    ("pandas",      "pandas"),
    ("numpy",       "numpy"),
    ("matplotlib",  "matplotlib"),
    ("requests",    "requests"),
    ("sqlalchemy",  "SQLAlchemy"),
    ("dotenv",      "python-dotenv"),
    ("statsmodels", "statsmodels"),
    ("sklearn",     "scikit-learn"),
]

faltan = []
for importar, instalar in PAQUETES:
    try:
        importlib.import_module(importar)
        print(f"  ok     {importar}")
    except ImportError:
        faltan.append(instalar)
        print(f"  FALTA  {importar}")

if faltan:
    print("\nInstalar con:\n  pip install " + " ".join(faltan))
    for paquete in faltan:
        print(f'Instalando {paquete}')
        %pip install {paquete}
else:
    print("\nTodos los paquetes del curso estan disponibles.")

  ok     pandas
  ok     numpy
  ok     matplotlib
  ok     requests
  ok     sqlalchemy
  ok     dotenv
  ok     statsmodels
  ok     sklearn

Todos los paquetes del curso estan disponibles.


---
## 2. Ubicar el kit de inicio

El kit es la plantilla oficial del trabajo semestral. Descomprime `kit-inicio-coyuntura.zip` y ajusta la variable `KIT` para que apunte a la carpeta resultante.

In [3]:
import os
os.getcwd()

'c:\\Users\\generico_fee\\Downloads\\kit-inicio-coyuntura\\notebooks'

In [4]:
from pathlib import Path

# Ajusta esta ruta si dejaste el kit en otro lugar.
KIT = Path("../../kit-inicio-coyuntura")

ESPERADO = [
    "requirements.txt",
    ".gitignore",
    ".env.example",
    "sql/01_esquema.sql",
    "sql/02_consultas_ejemplo.sql",
    "src/config.py",
    "src/extraccion.py",
    "src/carga_db.py",
    "src/exportar_powerbi.py",
    "notebooks/01_pipeline_datos.ipynb",
    "notebooks/02_analisis_ejemplo.ipynb",
]

if not KIT.exists():
    print(f"No encuentro el kit en: {KIT.resolve()}")
    print("Descomprime el zip y ajusta la variable KIT.")
else:
    print(f"Kit en: {KIT.resolve()}\n")
    for rel in ESPERADO:
        marca = "ok    " if (KIT / rel).exists() else "FALTA "
        print(f"  {marca} {rel}")

Kit en: C:\Users\generico_fee\Downloads\kit-inicio-coyuntura

  ok     requirements.txt
  ok     .gitignore
  ok     .env.example
  ok     sql/01_esquema.sql
  ok     sql/02_consultas_ejemplo.sql
  ok     src/config.py
  ok     src/extraccion.py
  ok     src/carga_db.py
  ok     src/exportar_powerbi.py
  ok     notebooks/01_pipeline_datos.ipynb
  ok     notebooks/02_analisis_ejemplo.ipynb


### Las tres tablas del proyecto

Antes de correr nada, mira el esquema. Son las tres tablas que vimos en clase.

In [5]:
print((KIT / "sql" / "01_esquema.sql").read_text(encoding="utf-8"))

-- ============================================================
-- Esquema de la base de datos del proyecto
-- Monitor de Coyuntura y Proyección Macroeconómica
-- Ciencia de Datos para Economía (ICO09425)
--
-- Motor: SQLite (compatible con PostgreSQL con cambios menores)
-- ============================================================

-- Tabla 1: Catálogo de series
-- Una fila por cada serie económica que usa el proyecto.
CREATE TABLE IF NOT EXISTS series (
    id_serie      TEXT PRIMARY KEY,      -- código en la fuente (ej: 'F073.TCO.PRE.Z.D' en BCCh, 'CPIAUCSL' en FRED)
    nombre        TEXT NOT NULL,         -- nombre legible (ej: 'Tipo de cambio observado')
    fuente        TEXT NOT NULL,         -- 'BCCH', 'FRED', 'INE', etc.
    frecuencia    TEXT NOT NULL,         -- 'D' diaria, 'M' mensual, 'Q' trimestral
    unidad        TEXT,                  -- 'CLP/USD', 'var. % interanual', 'índice', etc.
    id_tema       INTEGER REFERENCES temas(id_tema)
);

-- Tabla 2: Clasificación

---
## 3. Correr el pipeline en modo demo

El **modo demo** genera series sinteticas con la estructura real. Sirve para verificar que todo funciona antes de tener las credenciales del Banco Central y de FRED.

> **Antes de seguir:** el kit necesita `python-dotenv` incluso en modo demo. Si el chequeo de la seccion 1 lo marco como FALTA, instalalo ahora con `pip install python-dotenv`. Sin el, el error que aparece habla de un modulo `src` que no existe, y despista.

La funcion `correr` de abajo ejecuta un script del kit y muestra su salida.

In [6]:
import subprocess

def correr(*args):
    """Ejecuta un script del kit y muestra su salida."""
    proceso = subprocess.run(
        [sys.executable, *args],
        cwd=KIT,
        capture_output=True,
        text=True,
    )
    if proceso.stdout:
        print(proceso.stdout)
    if proceso.returncode != 0:
        print("ERROR (codigo", proceso.returncode, ")")
        print(proceso.stderr)
    return proceso.returncode

In [7]:
correr("src/carga_db.py", "--demo")

[OK] Esquema creado/verificado.
[OK] CatÃ¡logo cargado: 6 series.
Extrayendo series | modo: DEMO (datos sintÃ©ticos) | desde 2010-01-01 hasta hoy
  [OK] Tipo de cambio observado (CLP/USD): 3295 observaciones, hasta 2026-08-18
  [OK] Tasa de PolÃ­tica Monetaria (TPM): 3295 observaciones, hasta 2026-08-18
  [OK] IPC, variaciÃ³n mensual: 200 observaciones, hasta 2026-08-01
  [OK] Imacec (Ã­ndice): 200 observaciones, hasta 2026-08-01
  [OK] IPC Estados Unidos (Ã­ndice): 200 observaciones, hasta 2026-08-01
  [OK] Tasa Fed Funds (EE.UU.): 200 observaciones, hasta 2026-08-01
[OK] Observaciones cargadas/actualizadas: 7390.
Base de datos lista en: C:\Users\generico_fee\Downloads\kit-inicio-coyuntura\data\coyuntura.db



0

La senal de que funciono es la linea `Base de datos lista en: .../data/coyuntura.db`.

Ahora la vista que despues consumira Power BI.

In [11]:
correr("src/exportar_powerbi.py")

[OK] Vista 'vista_monitor' creada (7380 filas). Conecten Power BI a data/coyuntura.db.



0

---
## 4. Mirar la base por dentro

Ya tenemos una base de datos. Veamos que hay en ella.

In [8]:
import sqlite3
import pandas as pd

DB = KIT / "data" / "coyuntura.db"
print("Base  :", DB.resolve())
print("Tamano:", round(DB.stat().st_size / 1024, 1), "KB")

con = sqlite3.connect(DB)
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con)

Base  : C:\Users\generico_fee\Downloads\kit-inicio-coyuntura\data\coyuntura.db
Tamano: 844.0 KB


,name
0,series
1,temas
2,observaciones


In [11]:

pd.read_sql(""" SELECT * 
                FROM sqlite_master """, con)

,type,name,tbl_name,rootpage,sql
0,table,series,series,2,CREATE TABLE series (\n id_serie TEXT ...
1,index,sqlite_autoindex_series_1,series,3,None
2,table,temas,temas,4,CREATE TABLE temas (\n id_tema INTEGER ...
3,table,observaciones,observaciones,5,CREATE TABLE observaciones (\n id_serie ...
4,index,sqlite_autoindex_observaciones_1,observaciones,6,None
5,index,idx_obs_fecha,observaciones,7,CREATE INDEX idx_obs_fecha ON observaciones(fe...
6,view,vista_monitor,vista_monitor,0,CREATE VIEW vista_monitor AS\nSELECT o.id_seri...


```python 
# Seleccionar columna name
sqlite_master['name']

# Seleccionar columna name y tbl_name
sqlite_master[['name', 'tbl_name']]

# Seleccionar columna name y tbl_name con tbl_name igual a series 
sqlite_master[sqlite_master['tbl_name']=='series'][['name', 'tbl_name']]
```

In [12]:
# Seleccionar las columnas: name,	tbl_name
pd.read_sql(""" SELECT name, tbl_name 
                FROM sqlite_master """, con)

,name,tbl_name
0,series,series
1,sqlite_autoindex_series_1,series
2,temas,temas
3,observaciones,observaciones
4,sqlite_autoindex_observaciones_1,observaciones
5,idx_obs_fecha,observaciones
6,vista_monitor,vista_monitor


In [ ]:
# Seleccionar las columnas: name,	tbl_name
pd.read_sql(""" SELECT name, tbl_name 
                FROM sqlite_master 
                WHERE tbl_name = 'series' """, con)

# EN PYTHON sqlite_master[sqlite_master['tbl_name']=='series'][['name', 'tbl_name']]

,name,tbl_name
0,series,series
1,sqlite_autoindex_series_1,series


### El catalogo de series

La tabla `series` no guarda datos: guarda el **significado** de los datos. Cada fila declara de donde viene una serie, con que frecuencia se publica y en que unidad esta medida.

In [15]:
pd.read_sql("SELECT * FROM series", con)

,id_serie,nombre,fuente,frecuencia,unidad,id_tema
0,F073.TCO.PRE.Z.D,Tipo de cambio observado (CLP/USD),BCCH,D,CLP/USD,5
1,F022.TPM.TIN.D001.NO.Z.D,Tasa de Política Monetaria (TPM),BCCH,D,% anual,4
2,F074.IPC.VAR.Z.Z.C.M,"IPC, variación mensual",BCCH,M,var. % mensual,2
3,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec (índice),BCCH,M,índice 2018=100,1
4,CPIAUCSL,IPC Estados Unidos (índice),FRED,M,índice,6
5,FEDFUNDS,Tasa Fed Funds (EE.UU.),FRED,M,% anual,6


In [17]:
pd.read_sql(""" SELECT * 
                FROM series
                WHERE frecuencia = 'D' """, con)

,id_serie,nombre,fuente,frecuencia,unidad,id_tema
0,F073.TCO.PRE.Z.D,Tipo de cambio observado (CLP/USD),BCCH,D,CLP/USD,5
1,F022.TPM.TIN.D001.NO.Z.D,Tasa de Política Monetaria (TPM),BCCH,D,% anual,4


### La tabla de hechos

`observaciones` es la tabla que crece. Una fila por par (serie, fecha). Fijate en que **no** hay una columna por serie: agregar la serie numero trece no cambia el esquema, solo agrega filas.

In [18]:
pd.read_sql("SELECT * FROM observaciones ORDER BY fecha DESC LIMIT 10", con)

,id_serie,fecha,valor
0,F022.TPM.TIN.D001.NO.Z.D,2026-08-18,264.467
1,F073.TCO.PRE.Z.D,2026-08-18,231.051
2,F022.TPM.TIN.D001.NO.Z.D,2026-08-17,263.640
3,F073.TCO.PRE.Z.D,2026-08-17,231.089
4,F022.TPM.TIN.D001.NO.Z.D,2026-08-14,263.835
5,F073.TCO.PRE.Z.D,2026-08-14,230.887
6,F022.TPM.TIN.D001.NO.Z.D,2026-08-13,263.810
7,F073.TCO.PRE.Z.D,2026-08-13,230.936
8,F022.TPM.TIN.D001.NO.Z.D,2026-08-12,263.532
9,F073.TCO.PRE.Z.D,2026-08-12,230.984


### Tu primera consulta con JOIN

Esta consulta recorre las tres tablas a la vez. No hace falta entenderla en detalle todavia: es exactamente el contenido de la sesion del martes 11.

In [ ]:
consulta = """
SELECT * FROM temas
"""
pd.read_sql(consulta, con)

,id_tema,tema,descripcion
0,1,Actividad,Indicadores de actividad económica y producción
1,2,Precios,Inflación y sus componentes
2,3,Mercado laboral,"Empleo, desempleo y salarios"
3,4,Política monetaria,TPM y tasas de interés
4,5,Sector externo,"Tipo de cambio, términos de intercambio, commo..."
5,6,Internacional,"Indicadores de economías relevantes (EE.UU., C..."


In [ ]:
consulta = """
SELECT * FROM series
"""
pd.read_sql(consulta, con)

,id_serie,nombre,fuente,frecuencia,unidad,id_tema
0,F073.TCO.PRE.Z.D,Tipo de cambio observado (CLP/USD),BCCH,D,CLP/USD,5
1,F022.TPM.TIN.D001.NO.Z.D,Tasa de Política Monetaria (TPM),BCCH,D,% anual,4
2,F074.IPC.VAR.Z.Z.C.M,"IPC, variación mensual",BCCH,M,var. % mensual,2
3,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec (índice),BCCH,M,índice 2018=100,1
4,CPIAUCSL,IPC Estados Unidos (índice),FRED,M,índice,6
5,FEDFUNDS,Tasa Fed Funds (EE.UU.),FRED,M,% anual,6


In [ ]:
consulta = """
SELECT * FROM 
"""
pd.read_sql(consulta, con)

,id_serie,fecha,valor
0,F073.TCO.PRE.Z.D,2014-01-01,100.057
1,F073.TCO.PRE.Z.D,2014-01-02,100.122
2,F073.TCO.PRE.Z.D,2014-01-03,100.535
3,F073.TCO.PRE.Z.D,2014-01-06,100.707
4,F073.TCO.PRE.Z.D,2014-01-07,100.590
...,...,...,...
7385,FEDFUNDS,2026-04-01,107.332
7386,FEDFUNDS,2026-05-01,106.974
7387,FEDFUNDS,2026-06-01,105.324
7388,FEDFUNDS,2026-07-01,103.404


In [30]:
#  Unir tablas temas, series, observaciones
consulta = """
SELECT temas.tema, temas.descripcion, series.nombre, series.fuente, series.frecuencia, 
        series.unidad, observaciones.fecha, observaciones.valor
FROM temas
INNER JOIN series
ON temas.id_tema = series.id_tema
INNER JOIN observaciones
ON observaciones.id_serie = series.id_serie
"""

pd.read_sql(consulta, con)

,tema,descripcion,nombre,fuente,frecuencia,unidad,fecha,valor
0,Sector externo,"Tipo de cambio, términos de intercambio, commo...",Tipo de cambio observado (CLP/USD),BCCH,D,CLP/USD,2014-01-01,100.057
1,Sector externo,"Tipo de cambio, términos de intercambio, commo...",Tipo de cambio observado (CLP/USD),BCCH,D,CLP/USD,2014-01-02,100.122
2,Sector externo,"Tipo de cambio, términos de intercambio, commo...",Tipo de cambio observado (CLP/USD),BCCH,D,CLP/USD,2014-01-03,100.535
3,Sector externo,"Tipo de cambio, términos de intercambio, commo...",Tipo de cambio observado (CLP/USD),BCCH,D,CLP/USD,2014-01-06,100.707
4,Sector externo,"Tipo de cambio, términos de intercambio, commo...",Tipo de cambio observado (CLP/USD),BCCH,D,CLP/USD,2014-01-07,100.590
...,...,...,...,...,...,...,...,...
7385,Internacional,"Indicadores de economías relevantes (EE.UU., C...",Tasa Fed Funds (EE.UU.),FRED,M,% anual,2026-04-01,107.332
7386,Internacional,"Indicadores de economías relevantes (EE.UU., C...",Tasa Fed Funds (EE.UU.),FRED,M,% anual,2026-05-01,106.974
7387,Internacional,"Indicadores de economías relevantes (EE.UU., C...",Tasa Fed Funds (EE.UU.),FRED,M,% anual,2026-06-01,105.324
7388,Internacional,"Indicadores de economías relevantes (EE.UU., C...",Tasa Fed Funds (EE.UU.),FRED,M,% anual,2026-07-01,103.404


In [19]:
consulta = """
SELECT  t.tema,
        COUNT(DISTINCT s.id_serie) AS n_series,
        COUNT(o.valor)             AS n_observaciones
FROM        temas         AS t
LEFT JOIN   series        AS s  ON s.id_tema  = t.id_tema
LEFT JOIN   observaciones AS o  ON o.id_serie = s.id_serie
GROUP BY t.tema
ORDER BY n_observaciones DESC
"""

pd.read_sql(consulta, con)

,tema,n_series,n_observaciones
0,Sector externo,1,3295
1,Política monetaria,1,3295
2,Internacional,2,400
3,Precios,1,200
4,Actividad,1,200
5,Mercado laboral,0,0


Mira la fila **Mercado laboral**: aparece con `n_series = 0`. El catalogo del kit trae seis series y ninguna de empleo.

Esa fila esta ahi porque la consulta usa `LEFT JOIN`, que conserva todos los temas aunque no tengan series asociadas. Con `INNER JOIN` el tema desapareceria de la tabla y nadie notaria el hueco. La diferencia entre ver un cero y no ver la fila es exactamente el tipo de decision que veremos el martes 11.

> Tu proyecto necesita **al menos 12 series**. El kit trae seis: la mitad del catalogo esta por construirse, y ese es parte del Hito 1.

### Una serie, graficada

Sacar los datos de la base y graficarlos es el gesto que repetiremos todo el semestre.

In [ ]:
import matplotlib.pyplot as plt

serie = pd.read_sql("SELECT id_serie, nombre FROM series LIMIT 1", con).iloc[0]

datos = pd.read_sql(
    "SELECT fecha, valor FROM observaciones WHERE id_serie = ? ORDER BY fecha",
    con,
    params=(serie["id_serie"],),
    parse_dates=["fecha"],
).set_index("fecha")

ax = datos.plot(figsize=(9, 3.5), legend=False)
ax.set_xlabel("")
ax.set_ylabel(serie["nombre"])
plt.tight_layout()
plt.show()

print("Serie:", serie["id_serie"], "|", serie["nombre"], "| observaciones:", len(datos))

> **Cuidado con los niveles.** Estos son datos **sinteticos** del modo demo: la estructura es real, las cifras no. Si ves una TPM de 262 por ciento, el pipeline esta bien y el numero no significa nada. Los valores reales llegan cuando conectemos las credenciales, el 27 de agosto.

In [ ]:
con.close()
print("Conexion cerrada.")

---
## 5. Git desde hoy

La rubrica del trabajo asigna **10 puntos** a commits significativos de todos los integrantes distribuidos a lo largo del semestre. Un repositorio creado en noviembre no puede tener historia de agosto: ese puntaje se pierde y no se recupera.

In [ ]:
import shutil

if shutil.which("git") is None:
    print("Git no esta instalado.")
    print("  macOS  : xcode-select --install")
    print("  Windows: https://git-scm.com/download/win")
    print("  Linux  : sudo apt install git")
else:
    version = subprocess.run(["git", "--version"], capture_output=True, text=True).stdout.strip()
    nombre = subprocess.run(["git", "config", "--global", "user.name"], capture_output=True, text=True).stdout.strip()
    correo = subprocess.run(["git", "config", "--global", "user.email"], capture_output=True, text=True).stdout.strip()
    print(version)
    print("user.name :", nombre if nombre else "SIN CONFIGURAR")
    print("user.email:", correo if correo else "SIN CONFIGURAR")

Si aparece `SIN CONFIGURAR`, ejecuta esto **en la terminal**, una sola vez. Los commits sin identidad no cuentan como tuyos:

```bash
git config --global user.name  "Tu Nombre"
git config --global user.email "tu.correo@mail.udp.cl"
```

### Crear el repositorio del equipo

Una sola persona crea el repositorio en GitHub y agrega al resto como colaboradores. Despues, **cada integrante**:

```bash
git clone https://github.com/<equipo>/<repo>.git
cd <repo>

# copiar aqui el contenido del kit

git status                    # ver que cambio
git add .                     # preparar los cambios
git commit -m "Agrega el kit de inicio del proyecto"
git push
```

Un buen mensaje de commit describe **que cambio y por que**, no el archivo. `"Agrega series de actividad al catalogo"` sirve; `"cambios"` o `"update"` no.

### Autenticarse en GitHub desde un computador nuevo

Clonar un repositorio **publico** no pide credenciales. Hacer `push` si: GitHub necesita comprobar que la cuenta es tuya. Y desde 2021 **no acepta tu contrasena** en la terminal, aunque parezca que te la pregunta.

Este tramite se hace **una vez por computador**. Si trabajas en el laboratorio y tambien en tu casa, son dos veces.

Hay dos formas de identificarte:

| | HTTPS con token | SSH con llave |
| :--- | :--- | :--- |
| URL del repositorio | `https://github.com/...` | `git@github.com:...` |
| Que guarda el computador | un token | un par de llaves |
| Cuando conviene | computador prestado, uso corto | computador tuyo, uso permanente |

**La via corta resuelve las dos.** Instala GitHub CLI (<https://cli.github.com>) y ejecuta **en la terminal**:

```bash
gh auth login
```

Responde `GitHub.com`, luego `SSH`, acepta cuando ofrezca subir la llave publica, y elige `Login with a web browser`. Ese unico comando crea la llave si no existe, la registra en tu cuenta y deja lista de paso la autenticacion por HTTPS.

#### Si prefieres hacerlo a mano

```bash
# 1. Ver si este computador ya tiene una llave
ls ~/.ssh/id_ed25519.pub

# 2. Si no existe, crearla
ssh-keygen -t ed25519 -C "tu.correo@mail.udp.cl"

# 3. Cargarla en el agente, para no repetir la clave cada vez
ssh-add --apple-use-keychain ~/.ssh/id_ed25519   # macOS
ssh-add ~/.ssh/id_ed25519                        # Windows y Linux

# 4. Mostrar la llave PUBLICA y copiarla completa
cat ~/.ssh/id_ed25519.pub

# 5. Pegarla en https://github.com/settings/keys  ->  New SSH key

# 6. Comprobar
ssh -T git@github.com
```

La respuesta correcta del paso 6 es `Hi <usuario>! You've successfully authenticated`.

En macOS conviene ademas dejar escrito que llave usar, para que el sistema la recuerde despues de reiniciar. En el archivo `~/.ssh/config`:

```
Host github.com
  IdentityFile ~/.ssh/id_ed25519
  AddKeysToAgent yes
  UseKeychain yes
```

> **`Permission denied (publickey)` no significa que la llave este mal.** Significa que GitHub no la conoce: existe en tu disco, pero no esta registrada en tu cuenta. La solucion nunca es generar otra llave, es registrar la que ya tienes. Generar llaves nuevas una tras otra es el error mas comun frente a este mensaje.

> **La llave privada se trata como el `.env`.** El archivo `id_ed25519`, el que **no** termina en `.pub`, no se sube a GitHub, no se copia dentro del repositorio y no se manda por correo ni por WhatsApp. Lo unico que se comparte es la publica.

La celda siguiente informa en que estado esta **este** computador.

In [ ]:
# Esta celda se puede correr sola, sin ejecutar el resto del cuaderno.
import subprocess
import shutil
from pathlib import Path

def _run(cmd):
    return subprocess.run(cmd, capture_output=True, text=True)

print("=== Identidad de git ===")
for clave in ("user.name", "user.email"):
    valor = _run(["git", "config", "--global", clave]).stdout.strip()
    print(f"  {clave:11}: {valor if valor else 'SIN CONFIGURAR'}")

print("\n=== GitHub CLI ===")
if shutil.which("gh") is None:
    print("  gh no instalado  ->  https://cli.github.com")
else:
    estado = _run(["gh", "auth", "status"])
    print("  " + (estado.stdout or estado.stderr).strip().replace("\n", "\n  "))

print("\n=== Llaves SSH de este computador ===")
ssh = Path.home() / ".ssh"
publicas = sorted(ssh.glob("*.pub")) if ssh.exists() else []
if not publicas:
    print("  ninguna  ->  ssh-keygen -t ed25519 -C 'tu.correo@mail.udp.cl'")
for p in publicas:
    print(f"  {p.name}")

agente = _run(["ssh-add", "-l"])
print("  en el agente:", agente.stdout.strip() if agente.returncode == 0 else "ninguna cargada")

print("\n=== Respuesta de GitHub ===")
prueba = _run(["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=8", "-T", "git@github.com"])
salida = (prueba.stdout + prueba.stderr).strip()
print("  " + (salida.splitlines()[0] if salida else "sin respuesta"))

if "successfully authenticated" in salida:
    print("\n  Listo. GitHub reconoce la llave de este computador.")
else:
    print("\n  GitHub no reconoce ninguna llave de este computador.")
    print("  Ejecuta 'gh auth login' en la terminal, o registra la llave publica a mano.")

### La regla que no se negocia

El archivo `.env` contiene las claves del Banco Central y de FRED. **Nunca** se sube a GitHub. El `.gitignore` del kit ya lo excluye: no lo modifiques.

Un repositorio es publico y permanente. Una clave subida por error queda en el historial aunque se borre el archivo despues.

In [ ]:
gitignore = KIT / ".gitignore"

if gitignore.exists():
    contenido = gitignore.read_text(encoding="utf-8")
    print(contenido)
    print("-" * 40)
    print("Excluye .env :", ".env" in contenido)
else:
    print("No hay .gitignore. No subas nada hasta crear uno.")

---
## 6. Checklist de cierre

- [ ] Python 3.9 o superior verificado
- [ ] Paquetes del curso instalados, incluido `python-dotenv`
- [ ] Kit descomprimido y completo
- [ ] `carga_db.py --demo` corrio sin errores
- [ ] `coyuntura.db` creada, con las tres tablas pobladas
- [ ] Consulta con JOIN ejecutada
- [ ] Git instalado y con identidad configurada
- [ ] Autenticacion con GitHub resuelta: `ssh -T git@github.com` responde `Hi <usuario>!`
- [ ] Repositorio del equipo creado y clonado
- [ ] Primer commit propio registrado y subido
- [ ] `.gitignore` verificado: excluye `.env`

---
## 7. Para el martes 11 de agosto

1. **Conformar el equipo** de 3 a 4 integrantes y comunicarlo. Plazo: **lunes 10 de agosto**.
2. **Dejar el modo demo funcionando** en el computador de cada integrante, no solo en uno.
3. **Registrar un primer commit de cada integrante** en el repositorio del equipo.
4. **Iniciar el tramite de credenciales**, que tarda y se necesita el 27 de agosto:
   - Banco Central de Chile: <https://si3.bcentral.cl/Siete/es/Siete/API>
   - FRED: <https://fred.stlouisfed.org/docs/api/api_key.html>
5. **Instalar DB Browser for SQLite** y abrir `data/coyuntura.db` para mirar las tres tablas fuera de Python.

La proxima sesion es **SQL I**: las consultas que responden preguntas economicas sobre la base que hoy dejamos funcionando.